# Sistema de Scoring Crediticio con Optimización de Rentabilidad Ajustada por Riesgo

## Contexto del Problema

El riesgo crediticio representa uno de los principales desafíos para las entidades financieras y plataformas de préstamos. Cada vez que una institución otorga un crédito, asume la posibilidad de que el cliente no cumpla con sus obligaciones de pago, generando pérdidas financieras y afectando la rentabilidad del portafolio.

Este proyecto utiliza datos históricos de LendingClub, una plataforma estadounidense de *peer-to-peer lending (P2P lending)* que conectaba inversionistas con personas que solicitaban préstamos personales. Antes de aprobar un préstamo, LendingClub evaluaba el perfil financiero y crediticio de cada solicitante con el objetivo de estimar el riesgo asociado a la operación.

El dataset contiene información sobre millones de préstamos otorgados entre 2007 y 2018, incluyendo variables relacionadas con:

- Perfil financiero del solicitante
- Historial crediticio
- Condiciones del préstamo
- Estado final del crédito

La variable objetivo principal es `loan_status`, que permite identificar si un préstamo fue pagado correctamente o terminó en incumplimiento (*default*).

---

# Objetivo del Proyecto

El objetivo general de este proyecto es desarrollar un sistema de riesgo crediticio basado en Machine Learning capaz de estimar la probabilidad de incumplimiento de un préstamo utilizando información disponible al momento de la solicitud.

A partir de estas predicciones, el proyecto busca simular estrategias de aprobación y analizar cómo distintas políticas de riesgo afectan la rentabilidad y pérdida esperada de un portafolio de créditos.

---

# Objetivos Técnicos

Los objetivos técnicos del proyecto incluyen:

- Realizar un análisis exploratorio de datos (EDA) para comprender la estructura y calidad del dataset.
- Identificar y tratar valores faltantes, variables irrelevantes y posibles problemas de *data leakage*.
- Construir pipelines de preprocesamiento para variables numéricas y categóricas.
- Entrenar y comparar modelos de clasificación para predicción de default.
- Evaluar el desempeño utilizando métricas apropiadas para problemas de riesgo crediticio, como ROC-AUC, Recall y F1-score.
- Interpretar el comportamiento del modelo mediante técnicas de importancia de variables e interpretabilidad.

---

# Objetivos Aplicados al Negocio

Desde una perspectiva financiera, el proyecto busca transformar las probabilidades generadas por el modelo en herramientas de apoyo para la toma de decisiones.

Esto incluye:

- Estimar la probabilidad de default de cada solicitante.
- Construir un sistema de *credit scoring* basado en riesgo.
- Calcular la pérdida esperada (*Expected Loss*) de los préstamos.
- Simular políticas de aprobación utilizando distintos umbrales de riesgo.
- Optimizar decisiones de otorgamiento de crédito para maximizar rentabilidad y minimizar pérdidas.

De esta manera, el proyecto no se limita únicamente a un problema de clasificación, sino que incorpora conceptos reales de modelado de riesgo financiero y optimización de decisiones.

## 1.0 Librerías

In [27]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', None)


pd.set_option('display.max_columns', None)

In [28]:
df = pd.read_csv(r"C:\Users\camil\Documents\Laburo\Portafolio\credit-risk\data\raw\accepted_2007_to_2018Q4.csv", low_memory= False)

In [29]:
dict_df = pd.read_excel(r"C:\Users\camil\Documents\Laburo\Portafolio\credit-risk\data\raw\LCDataDictionary.xlsx")

c:\Users\camil\anaconda3\envs\mlp311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [30]:
browse_feat = dict_df["LoanStatNew"].dropna().values

In [31]:
browse_feat = [x.replace("\xa0", "") for x in browse_feat]

correcciones = {
    "is_inc_v": "verification_status",
    "verified_status_joint": "verification_status_joint"
}

browse_feat = [correcciones.get(col, col) for col in browse_feat]

In [32]:
valid_features = list(set(browse_feat).intersection(set(df.columns)))

In [33]:
len(valid_features)

76

In [34]:
df_browse = df[valid_features].copy()
df_browse["loan_status"] = df["loan_status"]

## 1.1 Entendimiento de los datos

In [35]:
df_browse.head()

,grade,dti_joint,fico_range_low,out_prncp,emp_length,last_fico_range_low,total_pymnt_inv,mths_since_last_record,pymnt_plan,emp_title,policy_code,tot_coll_amt,url,installment,member_id,purpose,id,collections_12_mths_ex_med,last_pymnt_amnt,revol_util,mths_since_rcnt_il,issue_d,total_bal_il,term,funded_amnt,total_pymnt,loan_amnt,all_util,dti,max_bal_bc,open_rv_24m,acc_now_delinq,revol_bal,recoveries,earliest_cr_line,mths_since_last_delinq,initial_list_status,loan_status,delinq_2yrs,open_il_24m,sub_grade,inq_last_12m,verification_status,annual_inc,inq_last_6mths,tot_cur_bal,last_fico_range_high,mths_since_last_major_derog,total_rec_int,title,int_rate,application_type,zip_code,next_pymnt_d,annual_inc_joint,addr_state,out_prncp_inv,total_cu_tl,open_acc_6m,home_ownership,verification_status_joint,inq_fi,collection_recovery_fee,total_rec_late_fee,funded_amnt_inv,last_credit_pull_d,last_pymnt_d,total_acc,desc,open_il_12m,fico_range_high,total_rec_prncp,open_rv_12m,il_util,open_acc,pub_rec
0,C,NaN,675.0,0.00,10+ years,560.0,4421.72,NaN,n,leadman,1.0,722.0,https://lendingclub.com/browse/loanDetail.acti...,123.03,NaN,debt_consolidation,68407277,0.0,122.67,29.7,21.0,Dec-2015,4981.0,36 months,3600.0,4421.723917,3600.0,34.0,5.91,722.0,3.0,0.0,2765.0,0.0,Aug-2003,30.0,w,Fully Paid,0.0,1.0,C4,4.0,Not Verified,55000.0,1.0,144904.0,564.0,30.0,821.72,Debt consolidation,13.99,Individual,190xx,NaN,NaN,PA,0.00,1.0,2.0,MORTGAGE,NaN,3.0,0.0,0.0,3600.0,Mar-2019,Jan-2019,13.0,NaN,0.0,679.0,3600.00,3.0,36.0,7.0,0.0
1,C,NaN,715.0,0.00,10+ years,695.0,25679.66,NaN,n,Engineer,1.0,0.0,https://lendingclub.com/browse/loanDetail.acti...,820.28,NaN,small_business,68355089,0.0,926.35,19.2,19.0,Dec-2015,18005.0,36 months,24700.0,25679.660000,24700.0,29.0,16.06,6472.0,3.0,0.0,21470.0,0.0,Dec-1999,6.0,w,Fully Paid,1.0,1.0,C1,6.0,Not Verified,65000.0,4.0,204396.0,699.0,NaN,979.66,Business,11.99,Individual,577xx,NaN,NaN,SD,0.00,0.0,1.0,MORTGAGE,NaN,0.0,0.0,0.0,24700.0,Mar-2019,Jun-2016,38.0,NaN,0.0,719.0,24700.00,2.0,73.0,22.0,0.0
2,B,13.85,695.0,0.00,10+ years,700.0,22705.92,NaN,n,truck driver,1.0,0.0,https://lendingclub.com/browse/loanDetail.acti...,432.66,NaN,home_improvement,68341763,0.0,15813.30,56.2,19.0,Dec-2015,10827.0,60 months,20000.0,22705.924294,20000.0,65.0,10.78,2081.0,2.0,0.0,7869.0,0.0,Aug-2000,NaN,w,Fully Paid,0.0,4.0,B4,1.0,Not Verified,63000.0,0.0,189699.0,704.0,NaN,2705.92,NaN,10.78,Joint App,605xx,NaN,71000.0,IL,0.00,5.0,0.0,MORTGAGE,Not Verified,2.0,0.0,0.0,20000.0,Mar-2019,Jun-2017,18.0,NaN,0.0,699.0,20000.00,0.0,73.0,6.0,0.0
3,C,NaN,785.0,15897.65,10+ years,675.0,31464.01,NaN,n,Information Systems Officer,1.0,0.0,https://lendingclub.com/browse/loanDetail.acti...,829.90,NaN,debt_consolidation,66310712,0.0,829.90,11.6,23.0,Dec-2015,12609.0,60 months,35000.0,31464.010000,35000.0,45.0,17.06,6987.0,1.0,0.0,7802.0,0.0,Sep-2008,NaN,w,Current,0.0,1.0,C5,0.0,Source Verified,110000.0,0.0,301500.0,679.0,NaN,12361.66,Debt consolidation,14.85,Individual,076xx,Apr-2019,NaN,NJ,15897.65,1.0,1.0,MORTGAGE,NaN,0.0,0.0,0.0,35000.0,Mar-2019,Feb-2019,17.0,NaN,0.0,789.0,19102.35,1.0,70.0,13.0,0.0
4,F,NaN,695.0,0.00,3 years,700.0,11740.50,NaN,n,Contract Specialist,1.0,0.0,https://lendingclub.com/browse/loanDetail.acti...,289.91,NaN,major_purchase,68476807,0.0,10128.96,64.5,14.0,Dec-2015,73839.0,60 months,10400.0,11740.500000,10400.0,78.0,25.37,9702.0,7.0,0.0,21929.0,0.0,Jun-1998,12.0,w,Fully Paid,1.0,3.0,F1,3.0,Source Verified,104433.0,3.0,331730.0,704.0,NaN,1340.50,Major purchase,22.45,Individual,174xx,NaN,NaN,PA,0.00,1.0,1.0,MORTGAGE,NaN,2.0,0.0,0.0,10400.0,Mar-2018,Jul-2016,35.0,NaN,0.0,699.0,10400.00,4.0,84.0,12.0,0.0


In [36]:
print(f'The dataset has {df_browse.shape[0]} rows and {df_browse.shape[1]} columns.')

The dataset has 2260701 rows and 76 columns.


## 1.2 Selección y Comprensión de Variables

El dataset original de LendingClub contiene más de 150 variables. Sin embargo, muchas de ellas corresponden a información generada después de que el préstamo fue aprobado o durante su ciclo de vida. Estas variables no estarían disponibles en el momento real de evaluar a un solicitante y, por lo tanto, introducirían problemas de *data leakage* si fueran utilizadas para entrenar el modelo.

Para identificar las variables realmente disponibles para inversionistas y analistas en el momento de la solicitud del préstamo, se utilizó el diccionario oficial de datos de LendingClub (`LCDataDictionary.xlsx`), específicamente la hoja denominada *Browse Notes*, publicada por Wendy Kan.

A partir de esta referencia, se realizó una intersección entre:

- Las variables presentes en el dataset original
- Las variables documentadas como visibles para inversionistas en el diccionario de datos

Este proceso permitió reducir el conjunto inicial de variables y conservar únicamente aquellas relevantes para un escenario realista de predicción de riesgo crediticio.

Posteriormente, durante las etapas de análisis exploratorio y modelado, algunas variables adicionales serán eliminadas debido a:

- Altos porcentajes de valores faltantes
- Baja utilidad predictiva
- Cardinalidad excesiva
- Posibles problemas de fuga de información (*data leakage*)

No obstante, ciertas variables serán conservadas temporalmente durante el EDA con el fin de entender mejor el comportamiento financiero de los préstamos, incluso si posteriormente no son utilizadas para entrenar el modelo final.


## 1.2.1 Diccionario de Variables — Lending Club

A continuación se presenta la descripción técnica y funcional de las variables utilizadas en el proyecto de riesgo crediticio con Lending Club. Las variables se agrupan según su naturaleza y utilidad dentro del análisis exploratorio, modelado predictivo y evaluación financiera.

---

### 1. Variables de Identificación y Metadatos

| Variable | Descripción |
|---|---|
| `id` | Identificador único del préstamo. | 
| `member_id` | Identificador único del cliente en Lending Club. |
| `url` | URL asociada al préstamo dentro de Lending Club. | 

---

### 2. Información del Préstamo y Condiciones Financieras

| Variable | Descripción |
|---|---|
| `loan_amnt` | Monto total solicitado por el prestatario. |
| `funded_amnt` | Monto financiado por Lending Club. |
| `funded_amnt_inv` | Monto financiado por inversionistas. |
| `term` | Duración del préstamo (36 o 60 meses). |
| `int_rate` | Tasa de interés anual aplicada al préstamo. |
| `installment` | Cuota mensual que debe pagar el prestatario. |
| `grade` | Calificación crediticia general asignada por Lending Club. |
| `sub_grade` | Subcategoría detallada de riesgo dentro del grade. |
| `purpose` | Motivo declarado del préstamo. |
| `title` | Título descriptivo asignado por el prestatario. |
| `issue_d` | Fecha de emisión del préstamo. |
| `application_type` | Tipo de aplicación: individual o conjunta (*joint application*). |
| `initial_list_status` | Estado inicial del préstamo en el marketplace (Whole/Fractional). |

---

### 3. Información Laboral y Socioeconómica

| Variable | Descripción |
|---|---|
| `emp_title` | Cargo laboral reportado por el prestatario. |
| `emp_length` | Antigüedad laboral en años. |
| `home_ownership` | Tipo de tenencia de vivienda (RENT, OWN, MORTGAGE). |
| `annual_inc` | Ingreso anual reportado por el prestatario. |
| `verification_status` | Estado de verificación de ingresos. |
| `annual_inc_joint` | Ingreso anual combinado para aplicaciones conjuntas. |
| `verification_status_joint` | Estado de verificación de ingresos en aplicaciones conjuntas. |
| `zip_code` | Código postal parcial del prestatario. |
| `addr_state` | Estado de residencia del prestatario. |

---

### 4. Variables de Riesgo Crediticio e Historial Financiero

| Variable | Descripción |
|---|---|
| `dti` | Relación deuda-ingreso (*Debt-to-Income Ratio*). |
| `dti_joint` | Relación deuda-ingreso para aplicaciones conjuntas. |
| `fico_range_low` | Límite inferior del rango FICO del prestatario. |
| `fico_range_high` | Límite superior del rango FICO del prestatario. |
| `last_fico_range_low` | Último rango FICO inferior registrado. |
| `last_fico_range_high` | Último rango FICO superior registrado. |
| `delinq_2yrs` | Número de moras en los últimos 2 años. |
| `acc_now_delinq` | Número de cuentas actualmente en mora. |
| `pub_rec` | Número de registros públicos negativos (ej. bancarrota). |
| `open_acc` | Número de cuentas de crédito abiertas. |
| `total_acc` | Número total de cuentas de crédito en historial. |
| `revol_bal` | Balance total de crédito revolvente utilizado. |
| `revol_util` | Utilización del crédito revolvente (%). |
| `inq_last_6mths` | Número de consultas de crédito en los últimos 6 meses. |
| `inq_last_12m` | Número de consultas de crédito en los últimos 12 meses. |
| `earliest_cr_line` | Fecha de apertura de la línea de crédito más antigua. |
| `mths_since_last_delinq` | Meses desde la última mora registrada. |
| `mths_since_last_record` | Meses desde el último registro público negativo. |
| `mths_since_last_major_derog` | Meses desde el último evento derogatorio severo. |

---

### 5. Variables Derivadas de Crédito e Información Bancaria

| Variable | Descripción |
|---|---|
| `tot_cur_bal` | Balance total actual de todas las cuentas. |
| `tot_coll_amt` | Total de cuentas enviadas a cobranza. |
| `total_rev_hi_lim` | Límite total de crédito revolvente disponible. |
| `total_bal_il` | Balance total de líneas de crédito a plazos (*installment loans*). |
| `all_util` | Utilización total del crédito disponible. |
| `il_util` | Utilización de líneas de crédito a plazos. |
| `max_bal_bc` | Balance máximo en tarjetas bancarias. |
| `total_cu_tl` | Número total de líneas de crédito financiero abiertas. |

---

### 6. Variables Relacionadas con Apertura y Actividad de Crédito

| Variable | Descripción |
|---|---|
| `open_acc_6m` | Número de cuentas abiertas en los últimos 6 meses. |
| `open_il_12m` | Número de líneas de crédito a plazos abiertas en 12 meses. |
| `open_il_24m` | Número de líneas de crédito a plazos abiertas en 24 meses. |
| `open_rv_12m` | Número de líneas revolventes abiertas en 12 meses. |
| `open_rv_24m` | Número de líneas revolventes abiertas en 24 meses. |
| `open_il_6m` | Número de líneas de crédito a plazos abiertas en 6 meses. |
| `mths_since_rcnt_il` | Meses desde la apertura más reciente de una línea a plazos. |
| `inq_fi` | Número de consultas financieras personales recientes. |

---

### 7. Estado del Préstamo y Variables Post-Emisión (Potential Data Leakage)

Estas variables contienen información generada después de la aprobación del préstamo y, aunque son útiles para análisis financieros y EDA, no deben utilizarse para entrenar modelos predictivos de default.

| Variable | Descripción |
|---|---|
| `loan_status` | Estado final o actual del préstamo. |
| `pymnt_plan` | Indica si existe un plan de pagos especial. |
| `out_prncp` | Principal pendiente por pagar. |
| `out_prncp_inv` | Principal pendiente correspondiente a inversionistas. |
| `total_pymnt` | Total pagado por el prestatario. |
| `total_pymnt_inv` | Total pagado a inversionistas. |
| `total_rec_prncp` | Capital recuperado. |
| `total_rec_int` | Intereses recuperados. |
| `total_rec_late_fee` | Cargos por mora recuperados. |
| `recoveries` | Monto recuperado después de default. |
| `collection_recovery_fee` | Comisiones asociadas a recuperación de deuda. |
| `last_pymnt_d` | Fecha del último pago realizado. |
| `last_pymnt_amnt` | Monto del último pago realizado. |
| `next_pymnt_d` | Próxima fecha estimada de pago. |
| `last_credit_pull_d` | Última fecha de consulta crediticia realizada por Lending Club. |

---

### 8. Variables Especiales para Aplicaciones Conjuntas (Joint Applications)

Estas variables solo aplican a préstamos realizados por más de un prestatario y presentan altos porcentajes de valores faltantes debido a que la mayoría de préstamos son individuales.

| Variable | Descripción |
|---|---|
| `annual_inc_joint` | Ingreso anual combinado de solicitantes conjuntos. |
| `dti_joint` | Relación deuda-ingreso conjunta. |
| `verification_status_joint` | Estado de verificación de ingresos conjuntos. |

---

### Consideraciones Técnicas

- Algunas variables con altos porcentajes de valores faltantes fueron conservadas inicialmente para análisis exploratorio.
- Variables como `grade` y `sub_grade` representan sistemas internos de scoring desarrollados por Lending Club y serán comparadas posteriormente con los modelos de Machine Learning desarrollados en este proyecto.

## 1.3 Información general

In [37]:
df_browse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 76 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   grade                        object 
 1   dti_joint                    float64
 2   fico_range_low               float64
 3   out_prncp                    float64
 4   emp_length                   object 
 5   last_fico_range_low          float64
 6   total_pymnt_inv              float64
 7   mths_since_last_record       float64
 8   pymnt_plan                   object 
 9   emp_title                    object 
 10  policy_code                  float64
 11  tot_coll_amt                 float64
 12  url                          object 
 13  installment                  float64
 14  member_id                    float64
 15  purpose                      object 
 16  id                           object 
 17  collections_12_mths_ex_med   float64
 18  last_pymnt_amnt              float64
 19  

 - Con un primer acercamiento encontramos qué podemos eliminar variables sin valor predictivo, ni analítico como lo son:
     - `id`
     - `member_id`
     - `url`

- También se identificaron variables con posible *data leakage*.
  - Estas variables contienen información generada después de la emisión del préstamo.
  - Por tanto, no pueden utilizarse en un modelo de originación crediticia (*application credit scoring*).

- Entre las variables con *data leakage* se encuentran:
  - `recoveries`
  - `collection_recovery_fee`
  - `total_rec_prncp`
  - `total_rec_int`
  - `last_pymnt_amnt`
  - `last_pymnt_d`
  - `next_pymnt_d`

- Estas variables podrían inflar artificialmente el desempeño del modelo y serán excluidas del entrenamiento.


- Algunas variables del dataset se encuentran almacenadas en tipos de datos que no representan correctamente su naturaleza estadística o temporal, por lo que es necesario realizar conversiones antes del análisis y modelado.


- Conversión de variables temporales desde `object` a formato `datetime`, especialmente:
  - `issue_d`
  - `earliest_cr_line`
  - `last_pymnt_d`
  - `last_credit_pull_d`
  - `next_pymnt_d`

- Conversión de variables categóricas ordinales y estructuradas:
  - `term`: actualmente almacenada como texto (`"36 months"`, `"60 months"`), debe convertirse a formato numérico entero.
  - `emp_length`: contiene categorías textuales (`"10+ years"`, `"2 years"`, etc.) y debe transformarse a una representación ordinal numérica.




## 1.4 Análisis de datos faltantes y duplicados

In [38]:
# ============================================
# Eliminación de variables irrelevantes,
# con data leakage y exceso de valores faltantes
# ============================================

cols_to_drop = [

    # ------------------------
    # Variables sin valor predictivo
    # ------------------------
    "id",
    "member_id",
    "url",

    # ------------------------
    # Variables con data leakage
    # ------------------------
    "recoveries",
    "collection_recovery_fee",
    "total_rec_prncp",
    "total_rec_int",
    "last_pymnt_amnt",
    "last_pymnt_d",
    "next_pymnt_d",

    # ------------------------
    # Variables con más de 70% de missing
    # ------------------------
    "verification_status_joint",
    "dti_joint",
    "annual_inc_joint",
    "desc",
    "mths_since_last_record",
    "mths_since_last_major_derog"
]

# Eliminación de columnas
df_browse = df_browse.drop(columns=cols_to_drop, errors="ignore")

print(f"Dataset después de limpieza inicial: {df.shape}")

Dataset después de limpieza inicial: (2260701, 151)


In [39]:
missing_table = pd.DataFrame({
    "Valores_Faltantes": df_browse.isna().sum(),
    "Porcentaje": df_browse.isna().mean() * 100,
})

missing_table = missing_table.sort_values(by="Porcentaje", ascending=False)
missing_table

,Valores_Faltantes,Porcentaje
mths_since_last_delinq,1158535,51.246715
il_util,1068883,47.281042
mths_since_rcnt_il,909957,40.251099
all_util,866381,38.323555
open_acc_6m,866163,38.313912
inq_last_12m,866163,38.313912
total_cu_tl,866163,38.313912
total_bal_il,866162,38.313868
open_il_24m,866162,38.313868
max_bal_bc,866162,38.313868


In [40]:
df_browse.duplicated().sum()

32

- El dataset no presenta registros duplicados, lo cual indica una buena consistencia inicial de los datos.

- Se establecerá un umbral de eliminación del 70% de valores faltantes.
  - Las variables con un porcentaje igual o superior a este umbral serán eliminadas del análisis principal.
  - Esto se debe a que:
    - Su capacidad informativa es limitada.
    - La imputación podría introducir sesgos importantes.
    - Algunas corresponden a casos muy específicos o poco representativos.
    - Otras contienen información posterior a la aprobación del préstamo (*data leakage*).

  - Bajo este criterio, variables como:
    - `member_id`
    - `verification_status_joint`
    - `dti_joint`
    - `annual_inc_joint`
    - `desc`
    - `mths_since_last_record`
    - `mths_since_last_major_derog`
  
  serán excluidas del análisis principal.

- La presencia de valores faltantes puede deberse a distintas razones:
  - El solicitante no proporcionó la información.
  - La información no fue registrada correctamente.
  - El valor no aplicaba para ciertos clientes.
  - El valor faltante realmente representa ausencia de eventos financieros.
  - Cambios históricos en la captura de variables dentro de LendingClub.

- Dependiendo del comportamiento de los valores faltantes, se aplicarán distintas estrategias:
  - Eliminación de variables con:
    - porcentaje excesivo de nulos,
    - alta cardinalidad irrelevante,
    - o ausencia de valor predictivo.
  
  - Eliminación de observaciones:
    - En variables con porcentajes mínimos de valores faltantes (<1%), los registros afectados podrán eliminarse directamente debido al gran tamaño del dataset.
  
  - Imputación:
    - Se considerarán técnicas como:
      - media,
      - mediana,
      - moda,
      - o categorías como `"Unknown"` para variables categóricas.
  
  - Tratamiento del missing como información:
    - En ciertos casos, la ausencia de información puede estar asociada al perfil de riesgo del solicitante.
    - Por ejemplo, no tener historial crediticio podría representar un perfil financiero distinto.
    - En estos escenarios, el valor faltante podrá tratarse como una categoría adicional.

- Procederemos a hacer una investigación del comportamiendo y causa de los valores faltantes

### 1.4.1 Investigando los NAs

In [41]:
print("La distribución porcentual de la variable 'loan_status' es:\n")

print(
    (df_browse["loan_status"]
     .value_counts(normalize=True) * 100)
)

La distribución porcentual de la variable 'loan_status' es:

loan_status
Fully Paid                                             47.629771
Current                                                38.852100
Charged Off                                            11.879630
Late (31-120 days)                                      0.949587
In Grace Period                                         0.373164
Late (16-30 days)                                       0.192377
Does not meet the credit policy. Status:Fully Paid      0.087939
Does not meet the credit policy. Status:Charged Off     0.033663
Default                                                 0.001769
Name: proportion, dtype: float64
